---
title: "بخش هشتم: مباحث تکمیلی"
authors:
  - name: دکتر محمود امین طوسی
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
        country: ایران
    email: m.amintoosi@gmail.com
    github: mamintoosi
    url: https://mamintoosi.github.io/
    roles: [supervision]
  - name: فاطمه آبشنگ
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
    github: Abshang
  - name: محمد مهدی جعفری
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
---

In [ ]:
import os
os.makedirs("src", exist_ok=True)
os.makedirs("bin", exist_ok=True)

In [ ]:
# ============================================================
# انیمیشن هدر آرگومان‌های خط فرمان (argc و argv)
# فقط اجرا کنید و رد شوید
# ============================================================
import base64, bisect, io, math, os, zlib
from PIL import Image as PILImage, ImageChops, ImageDraw, ImageFont
from IPython.display import HTML, Image, display

# ---------------- تنظیمات ----------------
HERO_WORDS      = [8, 3, 3, 3]           # طول (نویسه) کلمه‌های خط فرمان: برنامه، و سه آرگومان
HERO_THEME      = "ink"                  # "ink" = سیاه‌وسفید | "light" = سبز فایل HTML | "dark" = سبز تم تیره
HERO_BG         = None                   # None = سفید (ink/light) / رنگ paper (dark)؛ یا مثلاً "#F5F7F4"
HERO_SCALE      = 2                      # کیفیت (۲ = ریتینا)
HERO_PAD        = 8                      # حاشیه‌ی اطراف (px)
HERO_LOOP       = True                   # True: تکرار بی‌نهایت | False: فقط یک بار
HERO_FINAL_HOLD = 3000                   # مکث روی حالت نهایی قبل از تکرار (ms)
HERO_ALIGN      = "right"                # فقط برای HERO_DISPLAY="html": "right" | "left" | "center"
HERO_DISPLAY    = "image"                # "image" = display(Image(...)) | "html" = تگ img داخل HTML
HERO_FRAME_MS   = 20                     # طول هر فریم GIF (ms)

_PALETTES = {
    "ink":   dict(accent="#1F2328", soft="#F3F4F5", ink="#1F2328", muted="#6E7781",
                  surface="#FFFFFF", paper="#FFFFFF"),
    "light": dict(accent="#0F766E", soft="#E3F3F0", ink="#16202B", muted="#5B6B7A",
                  surface="#FFFFFF", paper="#F5F7F4"),
    "dark":  dict(accent="#2DD4BF", soft="#12302E", ink="#E6EDF3", muted="#9AA9B8",
                  surface="#131E2A", paper="#0D1620"),
}

_FONT_B64 = {
    700: "eNptVFtsFFUY/s/sdrYshHZhdwuNtHvr9kJZtjOzM3stbdl2u7RJDa0UChaWsnRb9tIbUDAFH4zxQrylJAi+YMQQQ4wvJiQqEh+IGjVReTBg8BK8G5/UB0Nn/WZ2gUo8mznn/Oe/ff939vzEiGg1PU4G2pd4eGAwfqT/MBH/Ak4dicFHui7/c/lPyG9BbhkY3CxY3xz+nohlIO8bHOoc+q3xk2bov4bcP5ZLTdEqOgX9t5Cl8dTsFNZq6J/CWjmePXbwo8OWYSKulchwJZNOHWjduPtp6H7HJ2dwwD/KRuHvgezJ5ObmDc+z85A/hLwuWxhLGT+oOAt/qybnUvNTVEVB6Hs1vPlULm1WamWiCi3e8anC7FwxQyLya3gdpNXKycbucxcm91ZF/6o0GH7GCd384vWz2vrjdxEfKcW/uSbuNkSeOCoN+BnuqNfhvJMU8nNNeqTlg+knK6E13PV4YBi5ndy7VIEYzyAKUV9pRbUC6yAt1Ypl0ZaNbV2YrpKfQiUMhjus9a4N5+Vu6JVVINYIxWmQIvh1Ug+QRKkDsx5aR2XFzPTVCA8gwk+TQ8UiZn+xWPz8XuZa2k0FOkPX6BZbwxJshL3BvoH/CBKryGkAO8A7a3FaGpwWxpwj7DX1VdaqXsV3nbux5BW4iEpcBJZxMGoFoxaqp41E3S6vN2CRZFkU7HabxcrzjYKMA6/bxUO0Ky5vQNKUNivPWhOZUCiT6B3XZqm9XdK+DQ2eujqPxyzv72Lhrv0yVvUa1mxUYGEhGhXUa0L0Y69mU1fnRU2DIKEJqM1AHBAtpoAiWkTb0JX3Ly6+nGdLzz1xbGkxD7sI7GqAtFazU2Q5UALlbud0rFbeZHLaBm77fUF/ILrAItne7oPB7v4zTDj2Q9tAixiKyT3poDTaufVk8vS0dkcR1F6NzPXlukWrXUS1WmB3QHEGnDZTAJUiNM8aEhPhEyFJCgal7efVn9iZz7InwpNmabSjTQn6Q23K9OlD0+81zaRCHaMSYncCrQGxVxKlAs4AE5lNRHXsSfUdtv6U+uWnCy++zRbVPu8FdY+GpQdY1sDepWNpFO33oLh4k3ut0+Y0iSYwL+t42Mb0cX7BGNiMm5IcvlXHK9g59Sa3oS7ZvmvsMXN62NustCWEQG0Ne2n6+npfgOVSibEZ/Nu0F+REHhs5y1VLsrJWNNSUr7x84yVurXbWmizE3PWXnv2Kt8ZbIjv8sVyfEo8r+MyxQlKaEViNeslY7wge2JqYjEwmQpFEIhJKoKIoGHAgUyXui7ltbtBpOMcqf11gt4vcwclbR5ZeAZ4O4Fmn36p8H8/927Xaa2p0VCVQjY26gXzvxvkSyN7J8LbklKdZ9vmUFndhoOdkToq1NByNJ+R0d/eY0hOf97TEzG0j0a5Rp2gJe5rbhGZ3uFp07+3Yc+iQssm9xaI4drUHd/j9O4JbdjqC1e3uTUGNL1ThBD6dr9m1jTzvdv0Xg8GwjK+LOk/hYX8sn+zLx1yOS6YyXWyoxFLvRCSW3ybOCuovbF6jSqOs9K4rqEvrAsYVyLta7wvY49GvglTa82CztryvwrmrvK/G6/HRdkrTOB2mLKVohoYhTUDO0BziFnB6gJLYp7CboDFYFygHKV/uQBjFNAn/2x/RQdGHqnSNmR6iOmLxnv4hoDHqfsVlvdFMXmoiTtevzqbm8rTuAav7XblS739WffYQd3Q8M6eFmJhLZZmWzQhfvc8zHv1Z6/DE/VFihjhUXc75L/9LWM4=",
    400: "eNptVF1sG0UQnr2zz6nbJjGxnZJaDeckdn7c2vh85ySOfY5zsR3SpJaTqIkbXF+dNHaUNCFJf6ASquCloCL6UCRU8YAqQEgICaEC5VcRlQpvCKhUKBL/rRAqAonSB9ScmT2bJFTsaW9vZmdnvvlmb4AAQDWcAhZyidS+EeXY3qMA3GnUPpAYGYtd+vvSbZRfRbl934jXX/fE/h8BSAHl3Mho7+jN/JeNuH8V5Wx+Xl2EbXAG979HOTCjLi/iWov7p3Ctmpl79PCvq889C8B4ANjVwrQ65fEceAr3buGUCqjgsmQWzzej3FyYXznBXiU/ofwpyjvmFvKq4bLxBTxvpfK8emIRaqAT95MU7xF1fnrLw/cXAYzU39LiwvJKqQACxs/RfaC5MpKhtups88Ganr+qWPYX1MA3X7xynq43fwjtBqV0h3EwN1DkgIHywHPsXQ1zZMZBgR7GoXvaPIiu2QpB5LFy4p5hYMbJWTCij6fRC8BgeSVZ8JMo0FBbNnnbNB6K4WsVfNBVxsDeJZ5/bRgXc13PzIi+MpDCmYQ4rmOIM4WPArprHZUV30RfDXgCEeFD5a5SCd++Uqn0+XrkBjgAC/A8XIFvyX0kQQ6Q17CmLPoHomFMFtlBvMsW3tLCWwjhM+Ql7QLxaKs4rzLX11x+JqQBE0LLFDJaxfwMFmiEDoB+p8slWgKSJPjtdpvFynFuv4QKV5OTQ9EedLrEAN20WTni6ctLUr5POUTfPQMDPXTu5PmdOM2i2kuisZwo5mLaaq8qDg2FSTSMb201PPS700HtHE7MCVFjTa+DGRGLgsUkBgWLYMtcufLyyQtpJnxu6uTaiTTa4S1iqhFpA7ULSpJYBtUUYXSsVs5k4m2T33UF+rqjAxmma7ovpgZigxfJ0ORv4YOecDwpo0LI9kYeG34vTWsUL93R+Wqs5C1Y7QJmSx03iUFe5G0mETNF1xzhY3kp0x+OKEpk8C3tK/Lk5aljwSmzmJXD/cnuQETJvJHOfLh1KReQsyL6TmGA2+h7K4Aq8iIRiE3A7Mh57W3CH9Ku/ZF58RnyuhZtfVObpFjGsA4s2jt1LG7Bvg7FyZma6ngbbxJMyLyk4yHtE0vmjDEalGU54u7enjGSU9o11uEYlCfyS+bJ4Q6vEklElDYnOZf+s8ErkYV8Qn0Ebxv9g6wYxwZ8JeuAFKwT2PpKySsVL3NrtZPd8WJ3Y+M7U58Z6mJtPeO+7plEdHg4itMcmo0HZx8kbu0i69jVeTgenw2lxuLxMTrLDDD1GKkK60V4WxPSyb5LjNqtDPlkjTk999H42uP0/pdv4A2sq7SBaKO+Vnt9vY6rDMvt1g2k9ZpzOkx6FRNDqscnS5LsbVNTyePT4cSejlwq6Z+U5Ul/ciTXtjth9o6HerMtXpvS5uvs8rUq1j0tB2MTxZQsdsRtXndG7kx7POlOeaLNa+1vl6KUMdrR8N7pjC3XuTmuyflfDGzdJsbe15kKTfi6C/1l7rZXCCN7KU8z/YliKFRMIHXa1yS7ThgOI8RoFzBswajVel/Ab/zpt6FU/jZs0tegvqHyXYt/jwvSMA0zcBTmQIUl2I9SEeUCrMAAThX1Rcij1QLMo3Sk0nlwlKbB/799ETsn9p8afccMDtgFRInvHcWaGvRzpU09kSJoBUbfr55TV47AjnusNrpxld73rHr8ZmCOzxRWqIviijpHaDQOM2vXbRhiwM5MOzGQjyuIKzrmgzIXKNdudOZ/AMM/XMo=",
}
_UPM, _ASC, _DESC = 1000, 1020, 300

# ---------------- ثابت‌های ابعاد (rem = 16px) ----------------
REM   = 16.0
RW    = 16.0 * REM               # عرض کل صحنه
RH    = 2.2 * REM                # ارتفاع کلمه‌ها در خط فرمان
TOP   = 0.4 * REM
GAP   = 6.0                      # فاصله‌ی بین کلمه‌ها / خانه‌ها
NSLOT = len(HERO_WORDS) + 1      # خانه‌های argv: argc خانه + یک خانه‌ی آخر (NULL)
SLOT_W = (RW - GAP * (NSLOT - 1)) / NSLOT
FY    = TOP + RH + 1.7 * REM     # بالای ردیف خانه‌ها
FH    = 1.6 * REM                # ارتفاع خانه‌ها
IDX_BASE = FY + FH + 1.3 * REM   # خط پایه‌ی شماره‌ی خانه‌ها
NUM_BASE = IDX_BASE + 2.2 * REM  # خط پایه‌ی عدد بزرگ argc
NUM_FONT, VAL_FONT, IDX_FONT = 1.7 * REM, 1.1 * REM, 0.95 * REM
STAGE_W  = RW
STAGE_H  = NUM_BASE + 0.4 * REM
_WSUM = sum(HERO_WORDS)
_WAVAIL = RW - GAP * (len(HERO_WORDS) - 1)


def _word_rect(k):
    x = sum(HERO_WORDS[:k]) / _WSUM * _WAVAIL + GAP * k
    return (x, TOP, HERO_WORDS[k] / _WSUM * _WAVAIL, RH)


def _slot_rect(k):
    return (k * (SLOT_W + GAP), FY, SLOT_W, FH)


def _rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def _bezier(x1, y1, x2, y2):
    def f(t, a, b):
        return 3 * a * t * (1 - t) ** 2 + 3 * b * t ** 2 * (1 - t) + t ** 3
    def ease(x):
        if x <= 0: return 0.0
        if x >= 1: return 1.0
        lo, hi = 0.0, 1.0
        for _ in range(40):
            mid = (lo + hi) / 2
            if f(mid, x1, x2) < x: lo = mid
            else: hi = mid
        return f((lo + hi) / 2, y1, y2)
    return ease


EASE_MOVE  = _bezier(.4, 0, .2, 1)
EASE_COLOR = _bezier(.25, .1, .25, 1)


class _Tween:
    def __init__(self, v, dur, ease):
        self.dur, self.ease = dur, ease
        self.segs = [(-1e9, tuple(v), tuple(v))]
        self.times = [-1e9]

    def val(self, t):
        i = bisect.bisect_right(self.times, t) - 1
        t0, a, b = self.segs[i]
        if t >= t0 + self.dur: return b
        if t <= t0: return a
        e = self.ease((t - t0) / self.dur)
        return tuple(x + (y - x) * e for x, y in zip(a, b))

    def set(self, v, now):
        v = tuple(v)
        if v != self.segs[-1][2]:
            self.segs.append((now, self.val(now), v))
            self.times.append(now)


def _simulate(pal):
    recs = []
    for k in range(len(HERO_WORDS)):
        recs.append(dict(k=k,
                         rect=_Tween(_word_rect(k), 520, EASE_MOVE),
                         reveal=_Tween((0.0,), 420, EASE_COLOR)))
    readout = [(-1e9, 0)]
    null_t = None
    t = 600
    for k, r in enumerate(recs):
        r["reveal"].set((1.0,), t)
        t += 420 + 380
        r["rect"].set(_slot_rect(k), t)
        readout.append((t + 380, k + 1))
        t += 520 + 300
    null_t = t
    t += 500
    return recs, readout, t, null_t


# ---------------- رسم یک فریم ----------------
_SS = 4
_K = HERO_SCALE * _SS
_fonts = {}
def _font(weight, css_px):
    key = (weight, css_px)
    if key not in _fonts:
        data = zlib.decompress(base64.b64decode(_FONT_B64[weight]))
        _fonts[key] = ImageFont.truetype(io.BytesIO(data), size=round(css_px * _K))
    return _fonts[key]


def _at(lst, t):
    return [v for tt, v in lst if tt <= t][-1]


def _draw(recs, readout, t, null_t, pal, bg, size):
    accent, soft, ink = _rgb(pal["accent"]), _rgb(pal["soft"]), _rgb(pal["ink"])
    surface, muted = _rgb(pal["surface"]), _rgb(pal["muted"])
    im = PILImage.new("RGB", (size[0] * _K, size[1] * _K), bg)
    d = ImageDraw.Draw(im)
    K = _K
    ox, oy = HERO_PAD, HERO_PAD
    r = lambda v: int(round(v * K))

    # خانه‌های خالی argv (طرح خاکستری)
    for k in range(NSLOT):
        x, y, w, h = _slot_rect(k)
        d.rectangle([r(ox + x), r(oy + y), r(ox + x + w) - 1, r(oy + y + h) - 1], fill=muted)
        d.rectangle([r(ox + x + 1), r(oy + y + 1), r(ox + x + w - 1) - 1, r(oy + y + h - 1) - 1], fill=bg)

    fills = [(accent, surface)] + [(soft, ink)] * (len(HERO_WORDS) - 1)
    for rec in recs:
        rv = rec["reveal"].val(t)[0]
        if rv <= 0.001:
            continue
        x, y, w, h = rec["rect"].val(t)
        big = h > 2 * REM
        bw = r(2 if big else 1)
        vis = x + w * rv
        bg_c, fg_c = fills[rec["k"]]
        xb_c = min(x + w, vis)
        X0, X1, Y0, Y1 = r(ox + x), r(ox + xb_c), r(oy + y), r(oy + y + h)
        if X1 - X0 < 1:
            continue
        d.rectangle([X0, Y0, X1 - 1, Y1 - 1], fill=accent)
        if X1 - X0 > 2 * bw and Y1 - Y0 > 2 * bw:
            d.rectangle([X0 + bw, Y0 + bw, X1 - bw - 1, Y1 - bw - 1], fill=bg_c)
        if big and rv >= 0.999:
            d.text(((X0 + X1) / 2, (Y0 + Y1) / 2 + r(0.36 * VAL_FONT)), str(rec["k"]),
                   font=_font(700, VAL_FONT), fill=fg_c, anchor="ms")

    # خانه‌ی آخر: NULL (خط مورب)
    if t >= null_t:
        x, y, w, h = _slot_rect(NSLOT - 1)
        d.line([r(ox + x + 4), r(oy + y + h - 4), r(ox + x + w - 4), r(oy + y + 4)],
               fill=ink, width=r(1.5))

    # شماره‌ی خانه‌ها زیر هر خانه (پررنگ وقتی پر شده)
    for k in range(NSLOT):
        x, y, w, h = _slot_rect(k)
        if k < len(HERO_WORDS):
            filled = t >= recs[k]["rect"].times[-1] + 520 if len(recs[k]["rect"].times) > 1 else False
        else:
            filled = t >= null_t
        d.text((r(ox + x + w / 2), r(oy + IDX_BASE)), str(k), font=_font(700, IDX_FONT),
               fill=ink if filled else muted, anchor="ms")

    d.text((r(ox + STAGE_W / 2), r(oy + NUM_BASE)), str(_at(readout, t)), font=_font(700, NUM_FONT),
           fill=ink, anchor="ms")
    return im.resize((size[0] * HERO_SCALE, size[1] * HERO_SCALE), PILImage.BOX)


def build_hero_gif():
    pal = _PALETTES[HERO_THEME]
    bg = _rgb(HERO_BG or ("#FFFFFF" if HERO_THEME in ("light", "ink") else pal["paper"]))
    size = (int(math.ceil(STAGE_W)) + 2 * HERO_PAD, int(math.ceil(STAGE_H)) + 2 * HERO_PAD)
    recs, readout, t_end, null_t = _simulate(pal)

    frames, durs, last_key, cache = [], [], None, {}
    t, t_stop = 0, t_end + 450
    while t <= t_stop:
        key = (tuple((tuple(round(v, 2) for v in rec["rect"].val(t)), round(rec["reveal"].val(t)[0], 3)) for rec in recs),
               _at(readout, t), t >= null_t,
               tuple(len(rec["rect"].times) > 1 and t >= rec["rect"].times[-1] + 520 for rec in recs))
        if key == last_key:
            durs[-1] += HERO_FRAME_MS
        else:
            if key not in cache: cache[key] = _draw(recs, readout, t, null_t, pal, bg, size)
            frames.append(cache[key]); durs.append(HERO_FRAME_MS); last_key = key
        t += HERO_FRAME_MS
    durs[-1] += HERO_FINAL_HOLD

    step = max(1, len(frames) // 16)
    sample = frames[::step]
    mosaic = PILImage.new("RGB", (frames[0].width * len(sample), frames[0].height))
    for i, fr in enumerate(sample): mosaic.paste(fr, (i * fr.width, 0))
    pal_img = mosaic.quantize(colors=256, method=PILImage.Quantize.MEDIANCUT, dither=PILImage.Dither.NONE)
    plt_ = pal_img.getpalette()[:768]
    bg_idx = min(range(len(plt_) // 3), key=lambda i: sum((plt_[i * 3 + k] - bg[k]) ** 2 for k in range(3)))
    plt_[bg_idx * 3:bg_idx * 3 + 3] = list(bg)
    pal_img = PILImage.new("P", (1, 1))
    pal_img.putpalette(plt_)
    solid = PILImage.new("RGB", frames[0].size, bg)
    q = []
    for fr in frames:
        qf = fr.quantize(palette=pal_img, dither=PILImage.Dither.NONE)
        same = ImageChops.difference(fr, solid).convert("L").point(lambda v: 255 if v == 0 else 0)
        qf.paste(bg_idx, mask=same)
        q.append(qf)

    buf = io.BytesIO()
    kw = dict(loop=0) if HERO_LOOP else {}
    q[0].save(buf, format="GIF", save_all=True, append_images=q[1:], duration=durs,
              disposal=1, optimize=False, **kw)
    return buf.getvalue(), size


hero_gif_bytes, hero_size = build_hero_gif()

os.makedirs("bin", exist_ok=True)
hero_gif_path = "bin/p08_hero_argv.gif"
with open(hero_gif_path, "wb") as f:
    f.write(hero_gif_bytes)

if HERO_DISPLAY == "image":
    display(Image(data=hero_gif_bytes, format="gif"))
else:
    _just = {"right": "flex-end", "left": "flex-start", "center": "center"}[HERO_ALIGN]
    display(HTML(
        f'<div style="display:flex;direction:ltr;justify-content:{_just}">'
        f'<img src="data:image/gif;base64,{base64.b64encode(hero_gif_bytes).decode()}" '
        f'width="{hero_size[0]}" height="{hero_size[1]}" alt="" style="max-width:100%;height:auto"></div>'
    ))


```{image} bin/p08_hero_argv.gif
:alt: انیمیشن قرار گرفتن آرگومان‌های خط فرمان در آرایه‌ی argv
:width: 296px
:align: right
:class: hero-anim
```

## مقدمه

تا اینجا هر برنامه‌ای که نوشتیم همه‌چیزش را خودش داشت: حدنصاب قبولی داخل کد نوشته می‌شد و ورودی‌ها را با `scanf` می‌گرفتیم. ابزارهای واقعی این‌طور کار نمی‌کنند. فرمان `gcc` را هر بار با فایل و گزینه‌های متفاوتی اجرا می‌کنیم و لازم نیست وسط اجرا منتظر تایپ ما بماند. در این بخش دو ابزار را می‌بینیم که چنین کاری را ممکن می‌کنند.

اولی **پیش‌پردازنده (preprocessor)** است. پیش از آنکه کامپایلر کار خودش را شروع کند، متن برنامه را اصلاح می‌کند: ثابت‌ها و ماکروها را جایگزین می‌کند و بخش‌هایی از کد را بسته به یک شرط نگه می‌دارد یا حذف می‌کند. با آن می‌توان بدون دست‌بردن در کد، یک نسخه‌ی اشکال‌زدایی و یک نسخه‌ی نهایی ساخت. دومی **آرگومان‌های خط فرمان** است، یعنی داده‌هایی که موقع اجرا کنار اسم برنامه می‌نویسیم و تابع `main` آن‌ها را با `argc` و `argv` می‌گیرد.

در هر دو فصل سناریوی **سیستم مدیریت نمرات** ادامه دارد. در پایان ابزار فیلتر نمراتی می‌سازیم که حدنصاب و نمره‌ها را از خط فرمان می‌گیرد و با یک سوئیچ کامپایل، پیام‌های اشکال‌زدایی هم چاپ می‌کند.

#### نقشه کلی بخش:

```text
source (.c) ──► preprocessor ──► compiler ──► executable
                #define  #ifdef  #include

terminal:  ./app  a  b  c ──► int main(int argc, char *argv[])
```

---

## فصل ۱۸ : Preprocessor

### 18.1 پیش‌پردازنده کجای کار است؟

در بخش اول دیدیم که `gcc` برنامه را در چند مرحله به فایل اجرایی تبدیل می‌کند. اولین مرحله **پیش‌پردازش** است و پیش از آنکه کامپایلر نوع‌ها و دستورها را بررسی کند انجام می‌شود. پیش‌پردازنده فقط خط‌هایی را اجرا می‌کند که با ‎`#`‎ شروع می‌شوند (‎`#include`‎، ‎`#define`‎، ‎`#ifdef`‎ و مانند آن‌ها) و بقیه‌ی متن را دست‌نخورده رد می‌کند. کارش **جایگزینی متنی** است و از متغیر، نوع داده یا اجرای برنامه خبر ندارد. مثلاً ‎`#include <stdio.h>`‎ فقط محتوای فایل هدر را به‌جای همان خط کپی می‌کند.

پیش‌پردازنده حدود سال ۱۹۷۳ به C اضافه شد. نسخه‌ی اول فقط ‎`#include`‎ و جایگزینی ساده داشت و ماکرو با پارامتر و کامپایل شرطی بعدتر به آن افزوده شدند.

خروجی پیش‌پردازنده را همیشه می‌توان دید: گزینه‌ی ‎`-E`‎ در `gcc` بعد از پیش‌پردازش متوقف می‌شود و متن حاصل را چاپ می‌کند. چند بار از آن استفاده می‌کنیم.

### 18.2 ثابت‌های نمادی با ‎`#define`‎

ساده‌ترین کاربرد ‎`#define`‎ دادن اسم به یک مقدار است تا عددهای بی‌توضیح (magic number) در کد پخش نشوند. هر جا اسم بیاید، پیش‌پردازنده متن جایگزین را می‌گذارد. این کار داخل رشته‌ها و توضیحات انجام نمی‌شود.

In [ ]:
%%writefile src/p08_c18_01_define_constant.c
#include <stdio.h>

#define MAX_STUDENTS 4
#define PASS_SCORE 10.0

int main(void)
{
    double scores[MAX_STUDENTS] = {18.5, 9.0, 14.0, 12.5};
    int passed = 0;

    for (int i = 0; i < MAX_STUDENTS; i++)
        if (scores[i] >= PASS_SCORE)
            passed++;

    printf("%d of %d passed\n", passed, MAX_STUDENTS);
    return 0;
}

Writing src/p08_c18_01_define_constant.c


In [ ]:
!gcc -Wall -o bin/p08_c18_01_define_constant src/p08_c18_01_define_constant.c && ./bin/p08_c18_01_define_constant

3 of 4 passed


برای دیدن چیزی که کامپایلر واقعاً می‌بیند، همین فایل را فقط پیش‌پردازش می‌کنیم و ته خروجی را نشان می‌دهیم (بالای آن محتوای ‎`stdio.h`‎ است):

In [ ]:
!gcc -E -P src/p08_c18_01_define_constant.c | tail -n 10

int main(void)
{
    double scores[4] = {18.5, 9.0, 14.0, 12.5};
    int passed = 0;
    for (int i = 0; i < 4; i++)
        if (scores[i] >= 10.0)
            passed++;
    printf("%d of %d passed\n", passed, 4);
    return 0;
}


اسم‌ها از بین رفته‌اند و جایشان `4` و ‎`10.0`‎ نشسته است.

:::{admonition} احتیاط با سمی‌کالن
:class: warning
بعد از ‎`#define`‎ سمی‌کالن نگذارید. متن جایگزین همه‌ی باقی‌مانده‌ی خط است، پس با ‎`#define MAX_STUDENTS 4;`‎ عبارت ‎`double scores[MAX_STUDENTS]`‎ به ‎`double scores[4;]`‎ تبدیل می‌شود و کامپایل خطا می‌دهد.
:::

ثابت‌ها را با `const` یا `enum` (بخش هفتم) هم می‌توان نوشت. آن‌ها نوع دارند و کامپایلر بررسی‌شان می‌کند، ولی ‎`#define`‎ فقط متن است. برای ثابت‌های ساده‌ی سراسری هر سه روش رایج‌اند.

### 18.3 ماکرو با پارامتر

ماکرو می‌تواند پارامتر بگیرد و شبیه فراخوانی تابع نوشته شود، ولی باز هم فقط متن را عوض می‌کند: پارامتر با **متنِ** آرگومان جایگزین می‌شود، نه با مقدار آن. این تفاوت در بند ۱۰.۷ بخش چهارم با `SQUARE_BAD` و ارزیابی چندباره‌ی آرگومان دقیق بررسی شد. اینجا فقط قاعده‌ی عملی را نگه می‌داریم: **هر پارامتر را داخل پرانتز بگذارید و کل عبارت را هم داخل پرانتز بگیرید.**

In [ ]:
%%writefile src/p08_c18_02_macro_param.c
#include <stdio.h>

#define IS_PASSED(score, limit) ((score) >= (limit))
#define MAX(a, b) ((a) > (b) ? (a) : (b))

int main(void)
{
    int a = 14, b = 19;

    printf("MAX(a, b)     = %d\n", MAX(a, b));
    printf("MAX(a + 6, b) = %d\n", MAX(a + 6, b));
    printf("IS_PASSED(a, 12)     = %d\n", IS_PASSED(a, 12));
    printf("IS_PASSED(a - 3, 12) = %d\n", IS_PASSED(a - 3, 12));
    return 0;
}

Writing src/p08_c18_02_macro_param.c


In [ ]:
!gcc -Wall -o bin/p08_c18_02_macro_param src/p08_c18_02_macro_param.c && ./bin/p08_c18_02_macro_param

MAX(a, b)     = 19
MAX(a + 6, b) = 20
IS_PASSED(a, 12)     = 1
IS_PASSED(a - 3, 12) = 0


پرانتزِ دور `a` باعث شد ‎`a + 6`‎ به‌عنوان یک مقدار کامل مقایسه شود. پرانتز بیرونی هم لازم است: وقتی ماکرو داخل عبارت بزرگ‌تری مثل ‎`10 + MIN(3, 5)`‎ بیاید، بدون آن اولویت عملگرها دوباره به هم می‌ریزد (تمرین ۲).

مشکل دومی هم هست که پرانتز حلش نمی‌کند: **اثر جانبی**. ماکرو آرگومان را به‌تعداد دفعات استفاده‌اش در متن کپی می‌کند. پس اگر ‎`i++`‎ به آن بدهیم، ‎`i++`‎ چند بار نوشته می‌شود:

In [ ]:
%%writefile src/p08_c18_03_macro_side_effect.c
#define SQUARE(x) ((x) * (x))

int i = 3;
int y = SQUARE(i++);

Writing src/p08_c18_03_macro_side_effect.c


In [ ]:
!gcc -E -P src/p08_c18_03_macro_side_effect.c

int i = 3;
int y = ((i++) * (i++));


عبارت ‎`((i++) * (i++))`‎ در یک عبارت دو بار `i` را تغییر می‌دهد و رفتار تعریف‌نشده (undefined behavior) دارد؛ ‎`gcc -Wall`‎ هم هنگام کامپایل هشدار می‌دهد. آرگومانی که اثر جانبی دارد، مثل ‎`i++`‎ یا فراخوانی تابع، نباید به ماکرو داده شود. برای کارهای کوچک معمولاً تابع معمولی بهتر است، چون نوع‌ها را بررسی می‌کند و آرگومانش فقط یک بار حساب می‌شود.

### 18.4 کامپایل شرطی

گاهی می‌خواهیم بخشی از کد فقط در شرایطی کامپایل شود؛ مثلاً پیام‌های اشکال‌زدایی را هنگام تست ببینیم ولی در نسخه‌ی نهایی اصلاً در فایل اجرایی نباشند. دستور ‎`#ifdef NAME`‎ بررسی می‌کند که `NAME` تعریف شده است یا نه. اگر شده باشد، خط‌ها تا ‎`#endif`‎ کامپایل می‌شوند و در غیر این صورت کنار گذاشته می‌شوند. دستور ‎`#ifndef`‎ عکس آن است و ‎`#else`‎ شاخه‌ی مقابل را می‌نویسد. هر ‎`#ifdef`‎ یا ‎`#ifndef`‎ باید با یک ‎`#endif`‎ بسته شود.

In [ ]:
%%writefile src/p08_c18_04_ifdef_debug.c
#include <stdio.h>

int main(void)
{
    int scores[] = {18, 14, 20};
    int sum = 0;

#ifndef DEBUG
    printf("Running in RELEASE mode.\n");
#endif

    for (int i = 0; i < 3; i++) {
        sum += scores[i];
#ifdef DEBUG
        printf("[DEBUG] i=%d, scores[i]=%d, sum=%d\n", i, scores[i], sum);
#endif
    }

    printf("Average: %.2f\n", sum / 3.0);
    return 0;
}

Writing src/p08_c18_04_ifdef_debug.c


اسم `DEBUG` در خود برنامه تعریف نشده است؛ آن را موقع کامپایل با گزینه‌ی ‎`-D`‎ تعریف می‌کنیم. اول بدون آن:

In [ ]:
!gcc -Wall -o bin/p08_c18_04_ifdef_debug src/p08_c18_04_ifdef_debug.c && ./bin/p08_c18_04_ifdef_debug

Running in RELEASE mode.
Average: 17.33


حالا با ‎`-DDEBUG`‎، بدون تغییر حتی یک خط از برنامه:

In [ ]:
!gcc -Wall -DDEBUG -o bin/p08_c18_04_ifdef_debug src/p08_c18_04_ifdef_debug.c && ./bin/p08_c18_04_ifdef_debug

[DEBUG] i=0, scores[i]=18, sum=18
[DEBUG] i=1, scores[i]=14, sum=32
[DEBUG] i=2, scores[i]=20, sum=52
Average: 17.33


در اجرای دوم خط‌های ‎`[DEBUG]`‎ چاپ می‌شوند و در اجرای اول اصلاً در فایل اجرایی نیستند، چون پیش‌پردازنده آن بخش را پیش از کامپایل حذف کرده است.

گزینه‌ی ‎`-D`‎ می‌تواند مقدار هم بدهد (‎`-DNAME=مقدار`‎). با ‎`#ifndef`‎ می‌توان برای یک ثابت مقدار پیش‌فرض گذاشت که از خط فرمان قابل تغییر باشد:

In [ ]:
%%writefile src/p08_c18_05_default_value.c
#include <stdio.h>

#ifndef PASS_SCORE
#define PASS_SCORE 10
#endif

int main(void)
{
    int score = 11;

    printf("limit = %d, score = %d: %s\n", PASS_SCORE, score,
           score >= PASS_SCORE ? "pass" : "fail");
    return 0;
}

Writing src/p08_c18_05_default_value.c


In [ ]:
!gcc -Wall -o bin/p08_c18_05_default_value src/p08_c18_05_default_value.c && ./bin/p08_c18_05_default_value

limit = 10, score = 11: pass


In [ ]:
!gcc -Wall -DPASS_SCORE=12 -o bin/p08_c18_05_default_value src/p08_c18_05_default_value.c && ./bin/p08_c18_05_default_value

limit = 12, score = 11: fail


وقتی مقدار را از بیرون ندهیم، ‎`#ifndef`‎ ثابت را ۱۰ می‌کند. با ‎`-DPASS_SCORE=12`‎ تعریف از بیرون می‌آید و ‎`#ifndef`‎ دیگر آن را عوض نمی‌کند.

### 18.5 محافظ هدر (include guard)

وقتی برنامه بزرگ می‌شود، تعریف‌ها (مثل `struct`) را در فایل هدر (‎`.h`‎) می‌گذاریم و با ‎`#include "name.h"`‎ به برنامه می‌آوریم. کوتیشن برای هدرهای خودمان است و ‎`<>`‎ برای هدرهای کتابخانه. مشکل وقتی پیش می‌آید که یک هدر دو بار وارد شود، مثلاً چون دو هدر دیگر هر دو آن را include کرده‌اند. آن‌وقت تعریف `struct` دو بار می‌آید و کامپایلر خطا می‌دهد. راه حل استاندارد همان ‎`#ifndef`‎ است:

In [ ]:
%%writefile src/p08_c18_06_student.h
#ifndef P08_STUDENT_H
#define P08_STUDENT_H

struct Student {
    int id;
    double score;
};

#endif

Writing src/p08_c18_06_student.h


In [ ]:
%%writefile src/p08_c18_06_include_guard.c
#include <stdio.h>
#include "p08_c18_06_student.h"
#include "p08_c18_06_student.h"

int main(void)
{
    struct Student s = {4001, 18.5};

    printf("%d %.1f\n", s.id, s.score);
    return 0;
}

Writing src/p08_c18_06_include_guard.c


In [ ]:
!gcc -Wall -o bin/p08_c18_06_include_guard src/p08_c18_06_include_guard.c && ./bin/p08_c18_06_include_guard

4001 18.5


بار اول `P08_STUDENT_H` تعریف‌نشده است، پس محتوای هدر کامپایل می‌شود و اسم تعریف می‌شود. بار دوم اسم تعریف شده و کل هدر نادیده گرفته می‌شود. بدون آن سه خط، همین برنامه خطای ‎`redefinition of 'struct Student'`‎ می‌دهد. اسم محافظ را معمولاً از اسم فایل می‌سازند تا با فایل دیگری یکی نشود.

### 18.6 تمرین

**تمرین ۱.** خروجی برنامه‌ی زیر را بدون اجرا پیش‌بینی کنید و بگویید چرا ۲۰ نیست.

In [ ]:
%%writefile src/p08_c18_07_exercise_add_bad.c
#include <stdio.h>

#define ADD_BAD(a, b) a + b

int main(void)
{
    printf("%d\n", ADD_BAD(2, 3) * 4);
    return 0;
}

Writing src/p08_c18_07_exercise_add_bad.c


In [ ]:
!gcc -Wall -o bin/p08_c18_07_exercise_add_bad src/p08_c18_07_exercise_add_bad.c && ./bin/p08_c18_07_exercise_add_bad

:::{admonition} پاسخ
:class: dropdown
خروجی `14` است. پیش‌پردازنده ‎`ADD_BAD(2, 3) * 4`‎ را به ‎`2 + 3 * 4`‎ تبدیل می‌کند و چون ضرب اولویت دارد، اول ‎`3 * 4`‎ حساب می‌شود و حاصل ‎`2 + 12`‎ است. با ‎`#define ADD(a, b) ((a) + (b))`‎ خروجی `20` می‌شد.
:::

**تمرین ۲.** ماکروی ‎`#define MIN(a, b) a < b ? a : b`‎ را در عبارت ‎`10 + MIN(3, 5)`‎ جایگزین متنی کنید، نتیجه را حساب کنید و نسخه‌ی درستش را بنویسید.

:::{admonition} پاسخ
:class: dropdown
بسط متنی ‎`10 + 3 < 5 ? 3 : 5`‎ می‌شود. عملگر ‎`+`‎ اولویت بالاتری از ‎`<`‎ و ‎`?:`‎ دارد، پس شرط ‎`(10 + 3) < 5`‎ بررسی می‌شود. این شرط نادرست است و نتیجه `5` می‌شود، در حالی که پاسخ درست `13` است. نسخه‌ی درست پارامترها و کل عبارت را پرانتز می‌گیرد:

```c
#include <stdio.h>

#define MIN_BAD(a, b) a < b ? a : b
#define MIN(a, b) ((a) < (b) ? (a) : (b))

int main(void)
{
    printf("MIN_BAD: %d\n", 10 + MIN_BAD(3, 5));
    printf("MIN:     %d\n", 10 + MIN(3, 5));
    return 0;
}
```

خروجی:

```text
MIN_BAD: 5
MIN:     13
```
:::

**تمرین ۳.** برنامه‌ای بنویسید که میانگین چند نمره را حساب کند و اگر ماکروی `VERBOSE` موقع کامپایل تعریف شده بود، جمع جزئی بعد از هر نمره را هم چاپ کند.

:::{admonition} پاسخ
:class: dropdown
```c
#include <stdio.h>

int main(void)
{
    int scores[] = {18, 14, 20};
    int n = 3, sum = 0;

    for (int i = 0; i < n; i++) {
        sum += scores[i];
#ifdef VERBOSE
        printf("after %d: sum = %d\n", scores[i], sum);
#endif
    }

    printf("average = %.2f\n", (double)sum / n);
    return 0;
}
```

بدون ‎`-DVERBOSE`‎ فقط ‎`average = 17.33`‎ چاپ می‌شود. با ‎`gcc -DVERBOSE`‎ خروجی چنین است:

```text
after 18: sum = 18
after 14: sum = 32
after 20: sum = 52
average = 17.33
```
:::

**تمرین ۴.** در هر قطعه‌کد یک خطا وجود دارد. نوع خطا و اصلاحش را بگویید.

```c
/* snippet 1 */
#define SIZE 10;
int arr[SIZE];

/* snippet 2 */
#define DOUBLE(x) x * 2
int y = DOUBLE(1 + 2);

/* snippet 3 */
#ifdef DEBUG
printf("debug\n");
```

:::{admonition} پاسخ
:class: dropdown
- قطعه‌ی ۱، سمی‌کالن بعد از ‎`#define`‎: خط دوم به ‎`int arr[10;];`‎ تبدیل می‌شود و کامپایل خطا می‌دهد. باید نوشت ‎`#define SIZE 10`‎.
- قطعه‌ی ۲، پرانتز نداشتن پارامتر: ‎`DOUBLE(1 + 2)`‎ به ‎`1 + 2 * 2`‎ تبدیل می‌شود و `5` می‌دهد، نه `6`. نسخه‌ی درست ‎`#define DOUBLE(x) ((x) * 2)`‎ است.
- قطعه‌ی ۳، نبودن ‎`#endif`‎: هر ‎`#ifdef`‎ باید بسته شود و گرنه `gcc` خطای ‎`unterminated #ifdef`‎ می‌دهد. بعد از خط `printf` باید ‎`#endif`‎ نوشته شود.
:::

---

## فصل ۱۹ : Command-Line Arguments

### 19.1 چرا آرگومان خط فرمان؟

وقتی ‎`gcc -Wall -o app main.c`‎ را اجرا می‌کنیم، خودِ `gcc` هم برنامه‌ای است که اسم فایل‌ها و گزینه‌ها را از همین خط می‌گیرد. اگر برنامه‌ی ما هم بخواهد مثل ابزارهای واقعی رفتار کند، یعنی اسم فایل یا یک عدد را موقع اجرا بگیرد، دو راه دارد: با `scanf` از کاربر بپرسد یا از خط فرمان بخواند. راه دوم ساده‌تر است و برنامه را در اسکریپت و ترکیب با برنامه‌های دیگر هم قابل استفاده می‌کند، چون منتظر تایپ کردن کسی نمی‌ماند.

### 19.2 ساختار ‎`main(int argc, char *argv[])`‎

برای گرفتن آرگومان‌ها، `main` را با دو پارامتر می‌نویسیم:

```c
int main(int argc, char *argv[])
```

- `argc` (argument count) تعداد کلمه‌های خط فرمان است. اسم خودِ برنامه هم یکی از آن‌هاست، پس همیشه حداقل ۱ است.
- `argv` (argument vector) آرایه‌ای از اشاره‌گرهای ‎`char *`‎ است، یعنی آرایه‌ای از رشته‌ها (بخش پنجم). خانه‌ی ‎`argv[0]`‎ اسم برنامه است، ‎`argv[1]`‎ تا ‎`argv[argc - 1]`‎ آرگومان‌ها هستند و ‎`argv[argc]`‎ همیشه `NULL` است.

انیمیشن بالای صفحه همین را نشان می‌دهد: چهار کلمه‌ی خط فرمان یکی‌یکی در خانه‌های ‎`argv[0]`‎ تا ‎`argv[3]`‎ می‌نشینند، `argc` برابر ۴ می‌شود و خانه‌ی آخر `NULL` است. نوشتن ‎`char *argv[]`‎ و ‎`char **argv`‎ یکی است، چون آرایه‌ای که به تابع می‌رسد به اشاره‌گر تبدیل می‌شود (Array Decay، بخش ششم).

In [ ]:
%%writefile src/p08_c19_01_print_args.c
#include <stdio.h>

int main(int argc, char *argv[])
{
    printf("argc = %d\n", argc);
    for (int i = 0; i < argc; i++)
        printf("argv[%d] = %s\n", i, argv[i]);
    printf("argv[%d] is %s\n", argc, argv[argc] == NULL ? "NULL" : "not NULL");
    return 0;
}

Writing src/p08_c19_01_print_args.c


In [ ]:
!gcc -Wall -o bin/p08_c19_01_print_args src/p08_c19_01_print_args.c && ./bin/p08_c19_01_print_args ali 20 mashhad

argc = 4
argv[0] = ./bin/p08_c19_01_print_args
argv[1] = ali
argv[2] = 20
argv[3] = mashhad
argv[4] is NULL


اگر برنامه بدون آرگومان اجرا شود، فقط اسم خودش در `argv` است و `argc` برابر ۱ می‌شود:

In [ ]:
!./bin/p08_c19_01_print_args

argc = 1
argv[0] = ./bin/p08_c19_01_print_args
argv[1] is NULL


چیدمان اجرای اول چنین است. اسم برنامه همان چیزی است که در خط فرمان نوشتیم:

```text
argv ─► [ ● ] ─► "./bin/p08_c19_01_print_args"
        [ ● ] ─► "ali"
        [ ● ] ─► "20"
        [ ● ] ─► "mashhad"
        [NULL]
```

### 19.3 تبدیل رشته به عدد

همه‌ی عنصرهای `argv` رشته‌اند. اگر کاربر `12` بنویسد، برنامه رشته‌ی ‎`"12"`‎ را می‌گیرد، یعنی دو نویسه‌ی ‎`'1'`‎ و ‎`'2'`‎، و نمی‌تواند مستقیم با آن حساب کند. برای تبدیل از ‎`<stdlib.h>`‎ استفاده می‌کنیم: `atoi` برای `int` و `atof` برای `double`.

In [ ]:
%%writefile src/p08_c19_02_add_two.c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    if (argc != 3) {
        printf("usage: %s <a> <b>\n", argv[0]);
        return 1;
    }

    int a = atoi(argv[1]);
    int b = atoi(argv[2]);

    printf("%d + %d = %d\n", a, b, a + b);
    return 0;
}

Writing src/p08_c19_02_add_two.c


In [ ]:
!gcc -Wall -o bin/p08_c19_02_add_two src/p08_c19_02_add_two.c && ./bin/p08_c19_02_add_two 12 30

12 + 30 = 42


In [ ]:
!./bin/p08_c19_02_add_two 12

usage: ./bin/p08_c19_02_add_two <a> <b>


دقت کنید که پیش از خواندن ‎`argv[1]`‎ و ‎`argv[2]`‎ شرط `argc` را بررسی کردیم. اگر آرگومان کم باشد، ‎`argv[1]`‎ برابر `NULL` است و دادنش به `atoi` برنامه را می‌اندازد، و ‎`argv[2]`‎ حتی بیرون از آرایه است. قاعده این است: **اول `argc` را بسنجید، بعد `argv` را بخوانید.**

`atoi` یک ضعف دارد: برای ورودی نامعتبر مثل `abc` هم ۰ برمی‌گرداند و نمی‌شود فهمید ۰ واقعی بوده یا خطا. تابع `strtol` در همان کتابخانه این را حل می‌کند. آدرس یک اشاره‌گر (`end`) به آن می‌دهیم و آن، جایی را که خواندن عدد در آن متوقف شده در `end` می‌گذارد؛ این همان اشاره‌گر به اشاره‌گر بخش ششم است. اگر هیچ رقمی خوانده نشود یا چیزی بعد از عدد بماند، ورودی نامعتبر است:

In [ ]:
%%writefile src/p08_c19_03_parse_int.c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    if (argc != 2) {
        printf("usage: %s <number>\n", argv[0]);
        return 1;
    }

    char *end;
    long value = strtol(argv[1], &end, 10);

    if (end == argv[1] || *end != '\0') {
        printf("\"%s\" is not a valid integer\n", argv[1]);
        return 1;
    }

    printf("value = %ld\n", value);
    return 0;
}

Writing src/p08_c19_03_parse_int.c


In [ ]:
!gcc -Wall -o bin/p08_c19_03_parse_int src/p08_c19_03_parse_int.c && ./bin/p08_c19_03_parse_int 42

value = 42


In [ ]:
!./bin/p08_c19_03_parse_int 4x2

"4x2" is not a valid integer


### 19.4 مقایسه‌ی آرگومان‌ها

ابزارها معمولاً گزینه‌هایی مثل ‎`--help`‎ هم می‌پذیرند. چون ‎`argv[i]`‎ اشاره‌گر است، نوشتن ‎`argv[1] == "--help"`‎ آدرس‌ها را مقایسه می‌کند و نه محتوا را. برای مقایسه‌ی رشته‌ها باید از `strcmp` استفاده کرد (بخش پنجم)، مثلاً ‎`strcmp(argv[1], "--help") == 0`‎. این را در تمرین ۳ به کار می‌بریم.

### 19.5 ابزار فیلتر نمرات

حالا همه‌چیز را کنار هم می‌گذاریم. برنامه حدنصاب و چند نمره را از خط فرمان می‌گیرد، با ماکروی `IS_PASSED` قبولی را تشخیص می‌دهد و اگر `DEBUG` تعریف شده باشد، وضعیت هر نمره را هم چاپ می‌کند:

In [ ]:
%%writefile src/p08_c19_04_cli_filter.c
#include <stdio.h>
#include <stdlib.h>

#define IS_PASSED(score, limit) ((score) >= (limit))

int main(int argc, char *argv[])
{
    if (argc < 3) {
        printf("usage: %s <limit> <score1> [score2 ...]\n", argv[0]);
        return 1;
    }

    int limit = atoi(argv[1]);
    int passed = 0;

    for (int i = 2; i < argc; i++) {
        int score = atoi(argv[i]);
#ifdef DEBUG
        printf("[DEBUG] score=%d -> %s\n", score,
               IS_PASSED(score, limit) ? "pass" : "fail");
#endif
        if (IS_PASSED(score, limit))
            passed++;
    }

    printf("limit: %d | scores: %d | passed: %d\n", limit, argc - 2, passed);
    return 0;
}

Writing src/p08_c19_04_cli_filter.c


اجرای بدون آرگومان فقط راهنما را چاپ می‌کند:

In [ ]:
!gcc -Wall -o bin/p08_c19_04_cli_filter src/p08_c19_04_cli_filter.c && ./bin/p08_c19_04_cli_filter

usage: ./bin/p08_c19_04_cli_filter <limit> <score1> [score2 ...]


با حدنصاب ۱۲ و پنج نمره‌ی ۱۰، ۱۵، ۱۸، ۹ و ۱۲:

In [ ]:
!./bin/p08_c19_04_cli_filter 12 10 15 18 9 12

limit: 12 | scores: 5 | passed: 3


از این پنج نمره، سه نمره (۱۵ و ۱۸ و ۱۲) حداقل ۱۲ هستند. با ‎`-DDEBUG`‎ همین برنامه جزئیات را هم می‌گوید:

In [ ]:
!gcc -Wall -DDEBUG -o bin/p08_c19_04_cli_filter src/p08_c19_04_cli_filter.c && ./bin/p08_c19_04_cli_filter 12 10 15 18 9 12

[DEBUG] score=10 -> fail
[DEBUG] score=15 -> pass
[DEBUG] score=18 -> pass
[DEBUG] score=9 -> fail
[DEBUG] score=12 -> pass
limit: 12 | scores: 5 | passed: 3


### 19.6 تمرین

**تمرین ۱.** اگر برنامه‌ای با فرمان ‎`./app ali 20 mashhad`‎ اجرا شود، مقدار `argc` و محتوای هر خانه‌ی `argv` چیست؟

:::{admonition} پاسخ
:class: dropdown
مقدار `argc` برابر `4` است. خانه‌ها چنین‌اند: ‎`argv[0]`‎ برابر ‎`"./app"`‎، ‎`argv[1]`‎ برابر ‎`"ali"`‎، ‎`argv[2]`‎ برابر ‎`"20"`‎ و ‎`argv[3]`‎ برابر ‎`"mashhad"`‎. خانه‌ی ‎`argv[4]`‎ هم `NULL` است. توجه کنید که ‎`argv[2]`‎ رشته‌ی ‎`"20"`‎ است و عدد `20` نیست؛ برای حساب کردن با آن باید `atoi` یا `strtol` به کار برد.
:::

**تمرین ۲.** برنامه‌ای بنویسید که هر تعداد عدد صحیح از خط فرمان بگیرد و مجموعشان را چاپ کند. اگر هیچ عددی داده نشد، راهنمای استفاده را چاپ کند.

:::{admonition} پاسخ
:class: dropdown
```c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    if (argc < 2) {
        printf("usage: %s <n1> [n2 ...]\n", argv[0]);
        return 1;
    }

    int sum = 0;
    for (int i = 1; i < argc; i++)
        sum += atoi(argv[i]);

    printf("sum = %d\n", sum);
    return 0;
}
```

برای ‎`./sum 3 4 5`‎ خروجی ‎`sum = 12`‎ است. حلقه از `1` شروع می‌شود چون ‎`argv[0]`‎ اسم برنامه است.
:::

**تمرین ۳.** برنامه‌ای بنویسید که اگر اولین آرگومانش ‎`--help`‎ بود راهنما را چاپ کند و در غیر این صورت به اسمی که در آرگومان آمده سلام کند (و اگر آرگومانی نبود، به `world`). چرا نباید از ‎`==`‎ برای مقایسه استفاده کرد؟

:::{admonition} پاسخ
:class: dropdown
```c
#include <stdio.h>
#include <string.h>

int main(int argc, char *argv[])
{
    if (argc > 1 && strcmp(argv[1], "--help") == 0) {
        printf("usage: %s [--help] [name]\n", argv[0]);
        return 0;
    }

    printf("hello, %s\n", argc > 1 ? argv[1] : "world");
    return 0;
}
```

با ‎`./app --help`‎ خروجی ‎`usage: ./app [--help] [name]`‎ است و با ‎`./app Sara`‎ خروجی ‎`hello, Sara`‎. عبارت ‎`argv[1] == "--help"`‎ دو **آدرس** را مقایسه می‌کند: آدرس رشته‌ی آرگومان و آدرس رشته‌ی ثابت داخل برنامه. این دو تقریباً هیچ‌وقت برابر نیستند، حتی اگر متن یکسان باشد. `strcmp` نویسه‌ها را مقایسه می‌کند. شرط ‎`argc > 1`‎ هم باید اول بیاید تا ‎`argv[1]`‎ بدون آرگومان خوانده نشود.
:::

**تمرین ۴.** ابزاری بنویسید که سه آرگومان بگیرد: اسم فایل ورودی نمرات (هر نمره در یک خط)، حدنصاب قبولی و اسم فایل گزارش. ابزار با ماکروی ‎`IS_PASSED(score, limit)`‎ قبولی هر نمره را تشخیص دهد و در پایان تعداد کل نمره‌ها و تعداد قبول‌شده‌ها را در فایل گزارش بنویسد. اگر تعداد آرگومان‌ها برابر ۳ نبود یا فایل باز نشد، پیام خطا بدهد. اگر `DEBUG` تعریف شده بود، وضعیت هر نمره را هم در ترمینال چاپ کند.

:::{admonition} پاسخ
:class: dropdown
```c
#include <stdio.h>
#include <stdlib.h>

#define IS_PASSED(score, limit) ((score) >= (limit))

int main(int argc, char *argv[])
{
    if (argc != 4) {
        printf("usage: %s <input> <limit> <output>\n", argv[0]);
        return 1;
    }

    FILE *in = fopen(argv[1], "r");
    if (in == NULL) {
        printf("cannot open %s\n", argv[1]);
        return 1;
    }
    int limit = atoi(argv[2]);

    int score, total = 0, passed = 0;
    while (fscanf(in, "%d", &score) == 1) {
        total++;
        if (IS_PASSED(score, limit))
            passed++;
#ifdef DEBUG
        printf("[DEBUG] %d -> %s\n", score,
               IS_PASSED(score, limit) ? "pass" : "fail");
#endif
    }
    fclose(in);

    FILE *out = fopen(argv[3], "w");
    if (out == NULL) {
        printf("cannot open %s\n", argv[3]);
        return 1;
    }
    fprintf(out, "total: %d\npassed: %d\n", total, passed);
    fclose(out);
    return 0;
}
```

اگر فایل نمرات شامل ‎`18 14 20 9 12`‎ باشد (هر عدد در یک خط)، با ‎`./report scores.txt 12 report.txt`‎ فایل گزارش این‌طور می‌شود:

```text
total: 5
passed: 4
```

و با کامپایل ‎`-DDEBUG`‎ همین اجرا در ترمینال چاپ می‌کند:

```text
[DEBUG] 18 -> pass
[DEBUG] 14 -> pass
[DEBUG] 20 -> pass
[DEBUG] 9 -> fail
[DEBUG] 12 -> pass
```

شرط ‎`argc != 4`‎ یعنی اسم برنامه و سه آرگومان؛ برنامه را با `fscanf` و مقدار برگشتی‌اش می‌خوانیم (بخش هفتم).
:::

---

## جمع‌بندی بخش

این بخش دو ابزار را معرفی کرد: پیش‌پردازنده که متن برنامه را پیش از کامپایل شکل می‌دهد، و آرگومان‌های خط فرمان که برنامه را بدون تغییر در کد با ورودی‌های متفاوت قابل اجرا می‌کنند. در پایان هر دو را در ابزار فیلتر نمرات کنار هم گذاشتیم.

### فصل ۱۸: پیش‌پردازنده

- کار پیش‌پردازنده: جایگزینی متنی برای خط‌هایی که با ‎`#`‎ شروع می‌شوند، پیش از کامپایل. با ‎`gcc -E`‎ خروجی‌اش دیده می‌شود.
- ثابت و ماکرو: ‎`#define NAME مقدار`‎ بدون سمی‌کالن. ماکرو با پارامتر نوع و ارزیابی ندارد و فقط متن را کپی می‌کند، پس پارامترها و کل عبارت پرانتز می‌خواهند و آرگومان نباید اثر جانبی داشته باشد.
- کامپایل شرطی: ‎`#ifdef`‎، ‎`#ifndef`‎، ‎`#else`‎ و ‎`#endif`‎. با ‎`-DNAME`‎ یا ‎`-DNAME=مقدار`‎ بدون تغییر کد، رفتار برنامه عوض می‌شود.
- محافظ هدر: سه خط ‎`#ifndef`‎، ‎`#define`‎ و ‎`#endif`‎ که از تعریف دوباره‌ی هدر جلوگیری می‌کند.

### فصل ۱۹: آرگومان‌های خط فرمان

- امضای `main`: ‎`int main(int argc, char *argv[])`‎. `argc` تعداد کلمه‌هاست و حداقل ۱ است.
- آرایه‌ی `argv`: ‎`argv[0]`‎ اسم برنامه است، آرگومان‌ها تا ‎`argv[argc - 1]`‎ می‌آیند و ‎`argv[argc]`‎ برابر `NULL` است.
- همه‌ی آرگومان‌ها رشته‌اند. برای عدد از `atoi` و `atof` یا، اگر خطای ورودی مهم است، از `strtol` استفاده می‌کنیم.
- مقایسه‌ی رشته با `strcmp`، و بررسی `argc` پیش از خواندن `argv`.

### اشتباه‌های رایج

| اشتباه | چه می‌شود؟ | راه درست |
|:--|:--|:--|
| نوشتن ‎`#define SIZE 10;`‎ | ‎`int arr[10;]`‎ و خطای کامپایل | بدون سمی‌کالن |
| ماکرو بدون پرانتز مثل ‎`a + b`‎ | اولویت عملگرها نتیجه را خراب می‌کند | پرانتز دور پارامترها و کل عبارت |
| دادن ‎`i++`‎ به ماکرو | آرگومان چند بار اجرا می‌شود | آرگومان بدون اثر جانبی یا تابع معمولی |
| فراموش‌کردن ‎`#endif`‎ | خطای ‎`unterminated #ifdef`‎ | بستن هر ‎`#ifdef`‎ و ‎`#ifndef`‎ |
| هدر بدون محافظ | تعریف دوباره‌ی `struct` | سه خط ‎`#ifndef`‎ و ‎`#define`‎ و ‎`#endif`‎ |
| خواندن ‎`argv[1]`‎ بدون بررسی `argc` | کرش یا رفتار نامعلوم | اول `argc` را بسنجید |
| فرض کردن اینکه آرگومان عدد است | ‎`"20"`‎ رشته است نه عدد | تبدیل با `atoi` یا `strtol` |
| مقایسه با ‎`argv[1] == "--help"`‎ | آدرس‌ها مقایسه می‌شوند | ‎`strcmp(argv[1], "--help") == 0`‎ |
| اعتماد به `atoi` برای ورودی نامعتبر | برای `abc` هم ۰ برمی‌گرداند | `strtol` و بررسی `end` |

## نگاهی به یک دانشمند تأثیرگذار در این حوزه

در این بخش دیدیم برنامه‌ی C می‌تواند مثل یک ابزار یونیکسی از خط فرمان آرگومان بگیرد. مثال اصلی همین کار در کتابی آمده که نسل‌ها C را از آن یاد گرفته‌اند: «زبان برنامه‌نویسی C» از **برایان کرنیگان** (Brian Kernighan) و دنیس ریچی.

![برایان کرنیگان (Brian Kernighan)|276](images/brian-kernighan.jpg)

**برایان کرنیگان** (متولد ۱۹۴۲ در تورنتو، کانادا) مهندسی فیزیک را در دانشگاه تورنتو خواند (۱۹۶۴) و دکترای مهندسی برق را در دانشگاه پرینستون گرفت (۱۹۶۹). همان سال به آزمایشگاه‌های بل (Bell Labs) پیوست، جایی که **یونیکس** و C ساخته می‌شدند. نام «یونیکس» را هم او پیشنهاد کرد. خودش گفته است که در طراحی زبان C نقشی نداشته و آن زبان کار ریچی است. سهم او در C نوشتن کتاب است: «زبان برنامه‌نویسی C» را با ریچی در ۱۹۷۸ منتشر کرد، و به‌خاطر همین کتاب به C قدیمی «K&R C» می‌گویند. حرف K در آن اول اسم اوست.

کرنیگان کتاب‌ها و ابزارهای دیگری هم دارد. او با دو همکارش زبان **AWK** را ساخت (حرف K در AWK هم از اسم اوست)، و کتاب‌های «ابزارهای نرم‌افزاری» (۱۹۷۶، با پی. جی. پلاگر) و «محیط برنامه‌نویسی یونیکس» (۱۹۸۴، با راب پایک) را نوشت. بعدها استاد دانشگاه پرینستون شد.

ارتباط کار او با این بخش روشن است. در کتاب K&R بخشی (۵.۱۰) به آرگومان‌های خط فرمان اختصاص دارد و مثالش برنامه‌ی `echo` است که آرگومان‌هایش را همان‌طور که هستند چاپ می‌کند؛ شبیه برنامه‌ی `p08_c19_01_print_args` که همین بخش نوشتیم. ابزارهای کوچکی مثل `echo`، `grep` و `awk` همه با همین مدل کار می‌کنند: برنامه را با آرگومان‌های متفاوت اجرا می‌کنیم و آن‌ها را کنار هم می‌گذاریم. کتاب «محیط برنامه‌نویسی یونیکس» هم برای آموزش کار با همین ابزارها نوشته شده است.